# Decision Tree - Heart Disease Prediction
**Tac gia:** Tuan  
**Notebook:** 04_DT_Tuan.ipynb  

## Noi dung
1. Feature Selection (Correlation)
2. Train model (Custom Decision Tree, max_depth=5)
3. Danh gia model (Accuracy, Precision, Recall, F1, Confusion Matrix)
4. Hyperparameter Tuning (max_depth, min_samples_split)
5. So sanh voi sklearn DecisionTreeClassifier
6. Ket luan


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('..'))

from src.models.decision_tree import DecisionTree
from src.utils import (train_test_split, accuracy_score, confusion_matrix,
                       precision_score, recall_score, f1_score)
print('Import thanh cong!')


---
## Phan 1: Feature Selection (Correlation)
Dung **correlation matrix** de tim cac features co tuong quan cao voi `HeartDisease`.  
Loai bo cac features nhieu (tuong quan thap, gan 0).


In [ ]:
df = pd.read_csv('../data/heart_preprocessed.csv')
print('Shape:', df.shape)
df.head(3)


In [ ]:
corr = df.corr()['HeartDisease'].drop('HeartDisease').sort_values(key=abs, ascending=False)

print('=' * 50)
print('Correlation voi HeartDisease:')
print('=' * 50)
for feat, val in corr.items():
    bar = '#' * int(abs(val) * 30)
    print(f'  {feat:<25} {val:+.4f}  {bar}')

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr.values]
corr.plot(kind='barh', color=colors)
plt.axvline(x=0.05,  color='green', linestyle='--', label='Nguong +0.05')
plt.axvline(x=-0.05, color='green', linestyle='--', label='Nguong -0.05')
plt.title('Correlation cua cac Features voi HeartDisease', fontsize=14)
plt.xlabel('Pearson Correlation')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
THRESHOLD = 0.05
selected_features = corr[abs(corr) > THRESHOLD].index.tolist()
dropped_features  = corr[abs(corr) <= THRESHOLD].index.tolist()

print(f'Nguong loai nhieu: |correlation| <= {THRESHOLD}')
print(f'Features GIU LAI ({len(selected_features)}): {selected_features}')
print(f'Features LOAI BO ({len(dropped_features)}): {dropped_features}')

X = df[selected_features].values
y = df['HeartDisease'].values.astype(int)
print(f'X shape: {X.shape}')
print(f'y: 0={sum(y==0)}, 1={sum(y==1)}')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')


---
## Phan 2: Train Model (Custom Decision Tree)
Dung **Custom Decision Tree** tu viet voi tham so mac dinh `max_depth=5`.

**Thuat toan:**
- Entropy: H(S) = -sum p_i * log2(p_i)
- Information Gain: IG(S, A) = H(S) - sum (|Sv|/|S|) * H(Sv)
- Chon split co IG lon nhat tai moi node


In [ ]:
print('Training Custom Decision Tree (max_depth=5)...')
custom_dt = DecisionTree(max_depth=5, min_samples_split=2)
custom_dt.fit(X_train, y_train)
print('Training hoan thanh!')
print(f'  max_depth         = {custom_dt.max_depth}')
print(f'  min_samples_split = {custom_dt.min_samples_split}')
print(f'  n_classes         = {custom_dt.n_classes}')


---
## Phan 3: Danh Gia Model
Tinh day du cac metrics:
- **Accuracy** | **Precision** | **Recall** | **F1-Score** | **Confusion Matrix**


In [ ]:
y_pred = custom_dt.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
cm   = confusion_matrix(y_test, y_pred)

print('=' * 45)
print('  KET QUA - CUSTOM DECISION TREE (depth=5)')
print('=' * 45)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('=' * 45)
print(f'  Confusion Matrix:')
print(f'    TN={cm[0,0]}  FP={cm[0,1]}')
print(f'    FN={cm[1,0]}  TP={cm[1,1]}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No', 'Pred: Yes'],
            yticklabels=['True: No', 'True: Yes'],
            ax=axes[0])
axes[0].set_title('Confusion Matrix - Custom DT (max_depth=5)', fontsize=12)
axes[0].set_ylabel('Thuc te')
axes[0].set_xlabel('Du doan')

# Bar chart metrics
metrics_dict = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
bars = axes[1].bar(metrics_dict.keys(), metrics_dict.values(),
                   color=['#2ecc71', '#3498db', '#e67e22', '#9b59b6'])
for b, v in zip(bars, metrics_dict.values()):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                 f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Metrics - Custom DT (max_depth=5)', fontsize=12)
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


---
## Phan 4: Hyperparameter Tuning
Thu nhieu bo tham so `(max_depth, min_samples_split)` de tim **best accuracy**.


In [ ]:
max_depths       = [2, 3, 4, 5, 6, 7, 8, 10]
min_samples_list = [2, 5, 10, 20]
results = []

print(f'Thu {len(max_depths)*len(min_samples_list)} bo tham so...\n')
print(f'  {"max_depth":<12} {"min_samples":<14} {"Accuracy":<12} {"F1-Score"}')
print('-' * 52)

for md_val in max_depths:
    for ms_val in min_samples_list:
        dt_tmp = DecisionTree(max_depth=md_val, min_samples_split=ms_val)
        dt_tmp.fit(X_train, y_train)
        preds   = dt_tmp.predict(X_test)
        acc_val = accuracy_score(y_test, preds)
        f1_val  = f1_score(y_test, preds)
        results.append({'max_depth': md_val, 'min_samples_split': ms_val,
                        'accuracy': acc_val, 'f1': f1_val})
        print(f'  {md_val:<12} {ms_val:<14} {acc_val:.4f}       {f1_val:.4f}')

results_df = pd.DataFrame(results)


In [ ]:
best_row = results_df.loc[results_df['accuracy'].idxmax()]
BEST_DEPTH       = int(best_row['max_depth'])
BEST_MIN_SAMPLES = int(best_row['min_samples_split'])

print('\n' + '=' * 50)
print('  BEST HYPERPARAMETERS')
print('=' * 50)
print(f'  max_depth         = {BEST_DEPTH}')
print(f'  min_samples_split = {BEST_MIN_SAMPLES}')
print(f'  Best Accuracy     = {best_row["accuracy"]:.4f} ({best_row["accuracy"]*100:.2f}%)')
print(f'  Best F1-Score     = {best_row["f1"]:.4f}')
print('=' * 50)


In [ ]:
# Heatmap
pivot = results_df.pivot(index='max_depth', columns='min_samples_split', values='accuracy')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.5, linecolor='gray', ax=axes[0])
axes[0].set_title('Accuracy Heatmap - Hyperparameter Tuning', fontsize=12)
axes[0].set_xlabel('min_samples_split')
axes[0].set_ylabel('max_depth')

# Line plot
subset = results_df[results_df['min_samples_split'] == 2].sort_values('max_depth')
axes[1].plot(subset['max_depth'], subset['accuracy'], marker='o', color='#2980b9', label='Accuracy')
axes[1].plot(subset['max_depth'], subset['f1'],       marker='s', color='#e74c3c', linestyle='--', label='F1-Score')
axes[1].axvline(x=BEST_DEPTH, color='green', linestyle=':', label=f'Best depth={BEST_DEPTH}')
axes[1].set_title('Accuracy & F1 theo max_depth (min_samples=2)', fontsize=12)
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Phan 5: So Sanh voi Sklearn
So sanh **Custom Decision Tree** vs **sklearn DecisionTreeClassifier** tren cung dataset, cung tham so.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score  as sk_accuracy,
    precision_score as sk_precision,
    recall_score    as sk_recall,
    f1_score        as sk_f1,
    confusion_matrix as sk_cm
)

print(f'Tham so: max_depth={BEST_DEPTH}, min_samples_split={BEST_MIN_SAMPLES}')

custom_best = DecisionTree(max_depth=BEST_DEPTH, min_samples_split=BEST_MIN_SAMPLES)
custom_best.fit(X_train, y_train)
y_pred_custom = custom_best.predict(X_test)

sklearn_dt = DecisionTreeClassifier(
    max_depth=BEST_DEPTH, min_samples_split=BEST_MIN_SAMPLES,
    criterion='entropy', random_state=42
)
sklearn_dt.fit(X_train, y_train)
y_pred_sklearn = sklearn_dt.predict(X_test)

print('Training ca hai model hoan thanh!')


In [ ]:
comparison = {
    'Metric':     ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Custom DT':  [
        accuracy_score(y_test, y_pred_custom),
        precision_score(y_test, y_pred_custom),
        recall_score(y_test, y_pred_custom),
        f1_score(y_test, y_pred_custom)
    ],
    'Sklearn DT': [
        sk_accuracy(y_test, y_pred_sklearn),
        sk_precision(y_test, y_pred_sklearn),
        sk_recall(y_test, y_pred_sklearn),
        sk_f1(y_test, y_pred_sklearn)
    ]
}
cmp_df = pd.DataFrame(comparison)
cmp_df['Difference'] = cmp_df['Custom DT'] - cmp_df['Sklearn DT']

print('=' * 60)
print('  SO SANH CUSTOM DT vs SKLEARN DT')
print('=' * 60)
print(cmp_df.to_string(index=False, float_format='{:.4f}'.format))
print('=' * 60)


In [ ]:
# Bar chart so sanh
metrics_names  = comparison['Metric']
custom_scores  = comparison['Custom DT']
sklearn_scores = comparison['Sklearn DT']
x = np.arange(len(metrics_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].bar(x - width/2, custom_scores,  width, label='Custom DT',  color='#3498db', alpha=0.85)
bars2 = axes[0].bar(x + width/2, sklearn_scores, width, label='Sklearn DT', color='#e74c3c', alpha=0.85)
for b in bars1:
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for b in bars2:
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=9)
axes[0].set_ylim(0, 1.15)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].set_ylabel('Score')
axes[0].set_title(f'Custom DT vs Sklearn DT (depth={BEST_DEPTH})', fontsize=12)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Confusion Matrix song song
cm_custom  = confusion_matrix(y_test, y_pred_custom)
cm_sklearn = sk_cm(y_test, y_pred_sklearn)
diff_cm    = cm_custom.astype(int) - cm_sklearn.astype(int)
labels = [f'Custom:{c}\nSklearn:{s}' for c, s in zip(cm_custom.flatten(), cm_sklearn.flatten())]
labels = np.array(labels).reshape(2, 2)
sns.heatmap(cm_custom, annot=labels, fmt='', cmap='Blues',
            xticklabels=['Pred: No', 'Pred: Yes'],
            yticklabels=['True: No', 'True: Yes'],
            ax=axes[1])
axes[1].set_title(f'Confusion Matrix Comparison (depth={BEST_DEPTH})', fontsize=12)
axes[1].set_ylabel('Thuc te')
axes[1].set_xlabel('Du doan')

plt.tight_layout()
plt.show()


---
## Phan 6: Ket Luan


In [ ]:
custom_acc  = accuracy_score(y_test, y_pred_custom)
sklearn_acc = sk_accuracy(y_test, y_pred_sklearn)
custom_f1   = f1_score(y_test, y_pred_custom)
sklearn_f1  = sk_f1(y_test, y_pred_sklearn)

print('=' * 65)
print('  TONG KET KET QUA')
print('=' * 65)
print(f'  Custom DT  - Accuracy: {custom_acc:.4f} | F1: {custom_f1:.4f}')
print(f'  Sklearn DT - Accuracy: {sklearn_acc:.4f} | F1: {sklearn_f1:.4f}')
print()
winner = 'Custom DT' if custom_acc >= sklearn_acc else 'Sklearn DT'
diff   = abs(custom_acc - sklearn_acc)
print(f'  Model tot hon: {winner} (hon {diff:.4f} accuracy)')
print('=' * 65)

print('''
NHAN XET:
----------
1. Custom DT va Sklearn DT cho ket qua rat gan nhau vi cung dung
   thuat toan Entropy + Information Gain, cung tham so.

2. Su khac biet nho (neu co) do Sklearn toi uu hoa cach chon
   threshold (dung midpoint giua 2 gia tri lien tiep), trong khi
   Custom DT dung truc tiep cac gia tri unique.

3. Feature Selection bang correlation giup loai bo nhieu,
   giam overfitting va tang toc do training.

DE XUAT CAI THIEN:
-------------------
- Dung Cross-Validation (k-fold) de danh gia on dinh hon.
- Thu Random Forest (ensemble nhieu DT) de tang accuracy.
- Dung SMOTE neu du lieu mat can bang (imbalanced).
- Toi uu threshold selection trong Custom DT (dung midpoint)
  de gan hon voi sklearn.
''')
